# StarCoder2 Fine-tuning - 3 Epochs (15k subset)
Set `SETUP = "A"` or `"B"`, then Run All. Both use the SAME 15k samples (id-based) for a fair comparison.

**Before committing:** Internet ON, GPU ON. Inputs needed: `python-train-cot` dataset.

In [ ]:
# Force single GPU (avoids cuda:1 device-mismatch)
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"


In [ ]:
# Install Unsloth only (it pins compatible transformers/peft/trl)
!pip install -q unsloth


In [ ]:
# ===== CONFIG =====
SETUP = "A"            # "A" = direct,  "B" = chain-of-thought
NUM_EPOCHS = 3
MAX_SEQ_LEN = 1280
SUBSET_SIZE = 15000    # set to 100 for a quick test, then back to 15000
OUTPUT_DIR = f"outputs_setup_{SETUP.lower()}"
ADAPTER_DIR = f"setup_{SETUP.lower()}_adapters"
print(f"Setup {SETUP} | {NUM_EPOCHS} epochs | subset {SUBSET_SIZE}")


In [ ]:
# Load StarCoder2-3B in 4-bit via Unsloth
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "bigcode/starcoder2-3b",
    max_seq_length = MAX_SEQ_LEN,
    dtype = None,
    load_in_4bit = True,
)
print("Model loaded.")


In [ ]:
# Attach LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_alpha = 32,
    lora_dropout = 0.05,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 42,
)
print("LoRA attached.")


In [ ]:
# Load data with an id-based subset so BOTH setups use the SAME samples
import json, random
BASE = "/kaggle/input/datasets/ayushsrivastava1805/python-train-cot"

def load_jsonl(path):
    return [json.loads(l) for l in open(path, encoding="utf-8")]

data_A = load_jsonl(f"{BASE}/python_train_filtered.jsonl")
data_B = load_jsonl(f"{BASE}/python_train_cot.jsonl")

ids_A = {s["id"] for s in data_A}
ids_B = {s["id"] for s in data_B}
common = sorted(ids_A & ids_B)
print(f"Common ids in both files: {len(common)}")

random.seed(42)
chosen = set(random.sample(common, SUBSET_SIZE))

if SETUP == "A":
    data = [s for s in data_A if s["id"] in chosen]
else:
    data = [s for s in data_B if s["id"] in chosen]

print(f"Setup {SETUP}: {len(data)} samples | fields: {list(data[0].keys())}")


In [ ]:
# Format helpers (strip license header so model sees real code)
def strip_license_header(code_str):
    lines = code_str.split("\n")
    start = 0
    for i, line in enumerate(lines):
        s = line.strip()
        if s.startswith("#") or s == "" or s.startswith('\"\"\"') or s.startswith("'''"):
            start = i + 1
        else:
            break
    if start >= len(lines) - 2:
        return code_str
    return "\n".join(lines[start:])

def format_sample(s):
    oldf  = strip_license_header(s.get("oldf", ""))[:1500]
    patch = s.get("patch", "")[:800]
    msg   = s.get("msg", "")
    if SETUP == "A":
        text = f"### Code:\n{oldf}\n\n### Change:\n{patch}\n\n### Review Comment:\n{msg}{tokenizer.eos_token}"
    else:
        reasoning = s.get("reasoning", "")
        text = f"### Code:\n{oldf}\n\n### Change:\n{patch}\n\n### Reasoning:\n{reasoning}\n\n### Review Comment:\n{msg}{tokenizer.eos_token}"
    return {"text": text}

print("--- Example ---")
print(format_sample(data[0])["text"][:600])


In [ ]:
from datasets import Dataset
formatted = [format_sample(s) for s in data]
dataset = Dataset.from_list(formatted)
print(f"Dataset: {len(dataset)} samples")


In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 50,
        num_train_epochs = NUM_EPOCHS,
        learning_rate = 2e-4,
        fp16 = True,
        logging_steps = 25,
        save_steps = 500,
        save_total_limit = 8,
        output_dir = OUTPUT_DIR,
        optim = "adamw_8bit",
        seed = 42,
        lr_scheduler_type = "cosine",
        dataset_text_field = "text",
    ),
)
trainer.train()


In [ ]:
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f"Done. Adapters saved to '{ADAPTER_DIR}'")
